# Predict

Run prediction using the model output from [02_modelling.ipynb](./02_modelling.ipynb)



In [1]:
from pathlib import Path

import geopandas as gpd
import geoutils as gu
import numpy as np
import pandas as pd
import rasterio as rio
import xdem


import mnk.substrat as subkart
import mnk

In [2]:
res = subkart.features.RESOLUTION
nodata = 255
crs = "EPSG:25833"

In [3]:
classifier = subkart.utils.load_classifier()

# Predict for Norge

## Wave Exposure


In [4]:
bolge = mnk.sources.bolge_exposure()


## DEM 50

In [5]:
dem_norge = mnk.sources.dem_data()

## Predict for all of Norway in one pass

In [6]:
print("Loading depth point data...")
gdf_points = mnk.sources.depth_point_data()
print(f"Depth points: {len(gdf_points):,}")


Loading depth point data...


Depth points: 6,047,001


In [7]:
predict_file_unmapped = "predict_unmapped.tif"
prob_file = "3band_probability.tif"

print("Loading sea map for all of Norway...")
gdf_sea_map = mnk.sources.sea_map_basisdata()
gdf_sea_map = subkart.features.depth_preprocess(gdf_sea_map)
transform, out_shape, bounds = subkart.features.to_raster_shapes(gdf_sea_map, res=res)

print("Resampling DEM...")
dem = subkart.utils.resample_dem(dem_norge.crop(bounds), out_shape, transform, crs)

print("Reprojecting wave exposure...")
bolge_norge = bolge.reproject(
    crs=crs, res=res,
    bounds=dict(left=bounds[0], bottom=bounds[1], right=bounds[2], top=bounds[3]),
)

print("Building feature arrays...")
X, valid_attrs, out_shape, transform = subkart.features.build(
    dem, gdf_sea_map, bolge_norge, valid_mask=None, res=res, dtype=np.float32,
    gdf_points=gdf_points,
)

Loading sea map for all of Norway...


Resampling DEM...


Reprojecting wave exposure...


Building feature arrays...
Computing interpolated depth raster...


  Point-refined 120 wide-range polygons.


Preparing sea_avg_depth...


... (4 outputs left out)

Preparing sea_convexity...


/home/jovyan/marint-naturkart-niva/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing wave exposure...


Stacking feature arrays...


In [8]:
Y_pred = classifier.predict(X)
Y_prob = classifier.predict_proba(X)

# Prediction raster (classes 0=løsbunn, 1=blanding, 2=fastbunn)
pred_map = np.full(out_shape, nodata, dtype=np.uint8)
pred_map[valid_attrs] = Y_pred.astype(np.uint8)
pred_map_masked = np.ma.masked_equal(pred_map, nodata)
pred_raster = gu.Raster.from_array(pred_map_masked, transform=transform, crs=crs, nodata=nodata)
pred_raster.to_file(predict_file_unmapped)
print(f"Saved {predict_file_unmapped}")

# Three-band probability raster (band 1=P(løsbunn), band 2=P(blanding), band 3=P(fastbunn))
prob_nodata = np.float32(-9999)
prob_bands = np.full((len(classifier.classes_), *out_shape), prob_nodata, dtype=np.float32)
for band_idx in range(len(classifier.classes_)):
    prob_bands[band_idx][valid_attrs] = Y_prob[:, band_idx].astype(np.float32)
with rio.open(
    prob_file, "w",
    driver="GTiff",
    height=out_shape[0], width=out_shape[1],
    count=len(classifier.classes_),
    dtype=np.float32,
    crs=crs,
    transform=transform,
    nodata=prob_nodata,
    compress="deflate",
    tiled=True,
    blockxsize=512,
    blockysize=512,
) as dst:
    dst.write(prob_bands)
print(f"Saved {prob_file}")

Saved predict_unmapped.tif


Saved 3band_probability.tif


## Calibrate probabilities

Apply the per-class isotonic calibrators fitted in [02_modelling.ipynb](./02_modelling.ipynb)
to the 3-band probability raster. This remaps only the probabilities (renormalised to sum to 1
per pixel); the class raster `predict_unmapped.tif` keeps the raw argmax, so predicted classes
are unchanged. The calibrated raster is what downstream postprocessing (`Sannsynlighet`) uses.

In [9]:
calibrators = subkart.training.load_calibrators()
prob_file_calibrated = prob_file.replace(".tif", "_calibrated.tif")
subkart.utils.calibrate_probability_raster(prob_file, calibrators, prob_file_calibrated)

# Keep the raw 3band_probability.tif; downstream postprocessing reads the calibrated file.
prob_file = prob_file_calibrated
print(f"Saved calibrated probabilities to {prob_file} using {len(calibrators)} isotonic calibrators")

Saved calibrated probabilities to 3band_probability_calibrated.tif using 3 isotonic calibrators
